# Kolmogorov–Navier–Stokes: select a quarter-length forecast and save 101 frames

This notebook **does not run AWFNO or repeat the 20-unit spin-up**. It reuses the original verified 300-trajectory dataset and evolves its stored, post-spin-up solutions on the GPU with the same ETDRK4 pseudospectral method.

**Scientific decision first.** The diagnostic compares equal-duration intervals (0–0.24, 0.26–0.50, 0.50–0.74, 0.76–1.00) using **200 training + 30 validation trajectories only**, not the 70 test trajectories. It reports relative change and temporal correlations. A later time offset is not inherently more chaotic: the spin-up already lasts 20 physical units.

**New data output:** choose `START_TIME=0.25` (default) or 0.0, 0.5, 0.75, with physical horizon **0.25**, solver and saved interval **0.0025**, and **101 time slices**. Thus 100 prediction intervals. The source grid is 128×128, split 200/30/70, and input conditions are inherited from the verified parent data. `times` in the saved dataset refers to *relative forecast time*, while `absolute_times` and `forecast_start` capture the original post-spin-up physical offset.

**Numerical quality:** Before and during generation, compare the new half-step numerical trajectory to original Δt=0.005 trajectories at every overlapping saved time. Enforce finite fields, source-frame identity at the anchor, mean-vorticity checks, chunk checksums, final PyTorch reload and exact split integrity. This is a **restart consistency check**, not a new 20-unit spin-up convergence study.

The final dataset is saved to a **new Google Drive folder**. Existing verified data remain untouched. If the GPU session disconnects, rerun the notebook: validated chunk files can be reused.

In [ ]:
#@title 1. Attach GPU, mount Drive, load previously verified dataset
from google.colab import drive
from pathlib import Path
import os, math, json, gc, time, hashlib, shutil
import numpy as np
import torch
import matplotlib.pyplot as plt

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > GPU'
DEVICE=torch.device('cuda')
print('GPU:', torch.cuda.get_device_name(0))

PARENT=Path('/content/drive/MyDrive/Kolmogorov_NS_Re500_verified_gpu_v1')
PARENT_FILE=PARENT/'kolmogorov_dataset.pt'
PARENT_METADATA=PARENT/'metadata.json'
EXPECTED_PARENT_SHA='82786a4293ac29bf23756c2cc39188e3b7fbb84bd26ef47061415fe76103ebb3'
assert PARENT_FILE.exists() and PARENT_METADATA.exists(), 'Verified parent dataset is missing'
pm=json.loads(PARENT_METADATA.read_text())
assert pm.get('status')=='PASS' and pm.get('final_sha256')==EXPECTED_PARENT_SHA, (
    'The saved source verification metadata differs from the approved data.')
try:
    source=torch.load(PARENT_FILE,map_location='cpu',weights_only=True,mmap=True)
except TypeError:
    source=torch.load(PARENT_FILE,map_location='cpu',weights_only=True)
U0=source['U']
assert tuple(U0.shape)==(300,128,128,51)
assert len(source['times'])==51 and abs(float(source['times'][-1])-1)<1e-6
assert float(source['Re'])==500.
for key,lo,hi in [('train_indices',0,200),('val_indices',200,230),('test_indices',230,300)]:
    np.testing.assert_array_equal(source[key].numpy(),np.arange(lo,hi))

# Choose after examining the results from cell 2.
START_TIME=0.25 #@param {type:'number'}
assert START_TIME in (0.0,0.25,0.5,0.75), 'Choose 0, 0.25, 0.5 or 0.75'
HORIZON=0.25
DT=0.0025  # numerical solver step and stored snapshot spacing
OLD_SNAPSHOT_DT=0.02
N_STEPS=int(round(HORIZON/DT))
N_FRAMES=N_STEPS+1
assert N_FRAMES==101
BATCH_SIZE=16 #@param {type:'integer'}
assert 1<=BATCH_SIZE<=64
RUN_GENERATION=False #@param {type:'boolean'}
# First run the diagnostics. Set RUN_GENERATION=True after you select START_TIME.

NX=128
RE=500.
NU=1./RE
FORCING_MODE=4
DOMAIN=2*np.pi
T_SPINUP=20.
BASE_SEED=1234
IC_AMPLITUDE=1.
IC_FILTER_K0=6.
MAX_RECORDED_CFL=1.5
CFL_ADVISORY_THRESHOLD=0.9
REL_REPLAY_TOL=0.005

ANCHOR_IDX=int(np.floor(START_TIME/OLD_SNAPSHOT_DT+1e-10))
ANCHOR_TIME=ANCHOR_IDX*OLD_SNAPSHOT_DT
WARM_STEPS=int(round((START_TIME-ANCHOR_TIME)/DT))
assert abs(ANCHOR_TIME+WARM_STEPS*DT-START_TIME)<1e-10
assert 0<=WARM_STEPS<8
TIMES=np.arange(N_FRAMES,dtype=np.float64)*DT
ABSOLUTE_TIMES=START_TIME+TIMES
X=np.arange(NX,dtype=np.float64)*DOMAIN/NX
Y=X.copy()
OUTPUT_SHAPE=(300,NX,NX,N_FRAMES)
ROOT=Path('/content/drive/MyDrive')/(
    f'Kolmogorov_NS_Re500_start{START_TIME:g}_horizon{HORIZON:g}_dt{DT:g}_v1'
)
CHUNKS=ROOT/'chunks'
LOCAL=Path('/content/kolmogorov_ns_quarter_build')
if RUN_GENERATION:
    for folder in (ROOT,CHUNKS,LOCAL):folder.mkdir(parents=True,exist_ok=True)

SPEC={
    'schema':'Kolmogorov-NS-verified-parent-short-restart-v1',
    'parent_sha256':EXPECTED_PARENT_SHA,
    'grid':[NX,NX],'Re':RE,'nu':NU,'forcing_mode':FORCING_MODE,
    'spinup_time':T_SPINUP,'start_time':START_TIME,'horizon':HORIZON,
    'solver_dt':DT,'snapshot_dt':DT,'shape':list(OUTPUT_SHAPE),
    'split':[200,30,70],'parent_anchor_index':ANCHOR_IDX,
    'parent_anchor_time':ANCHOR_TIME,'warm_steps_from_anchor':WARM_STEPS,
    'batch_size':BATCH_SIZE,'method':'real-phi ETDRK4 2/3 Fourier pseudospectral',
}
SPEC_HASH=hashlib.sha256(json.dumps(SPEC,sort_keys=True).encode()).hexdigest()[:16]
print('Parent:',PARENT_FILE)
print('Quarter horizon:',START_TIME,'to',START_TIME+HORIZON,
      'snapshots:',N_FRAMES,'initial anchor:',ANCHOR_TIME)
print('New Drive folder:',ROOT)
print('Generation enabled:',RUN_GENERATION)


In [ ]:
#@title 2. Compare change and temporal correlation across the original trajectory
# Only TRAIN+VALIDATION trajectories are used to compare time intervals.
# The test trajectories remain sealed until after choosing the benchmark.
windows=[('0.00 -> 0.24',0,12),('0.26 -> 0.50',13,25),
         ('0.50 -> 0.74',25,37),('0.76 -> 1.00',38,50)]
population=range(230)
stats={}
for label,left,right in windows:
    rels=[];cors=[];norms=[]
    for i in population:
        a=U0[i,:,:,left].numpy().astype(np.float64)
        b=U0[i,:,:,right].numpy().astype(np.float64)
        rels.append(np.linalg.norm(b-a)/max(np.linalg.norm(a),1e-12))
        ac=a-a.mean();bc=b-b.mean()
        cors.append(np.sum(ac*bc)/max(np.linalg.norm(ac)*np.linalg.norm(bc),1e-12))
        norms.append(np.linalg.norm(a))
    stats[label]=(float(np.mean(rels)),float(np.std(rels)),
                  float(np.mean(cors)),float(np.percentile(rels,10)),
                  float(np.percentile(rels,90)))
print('All intervals have duration 0.24; train+validation only (N=230).')
print(f'{"Interval":<18} {"mean relative change":>21} {"std":>10} {"mean correlation":>19} {"10–90% change":>20}')
for label,(mean,std,corr,p10,p90) in stats.items():
    print(f'{label:<18} {mean:>21.4f} {std:>10.4f} {corr:>19.4f} {p10:.3f}–{p90:.3f}')
fig,axes=plt.subplots(1,2,figsize=(12,3.8),constrained_layout=True)
labels=list(stats)
axes[0].bar(labels,[stats[x][0] for x in labels],
            yerr=[stats[x][1] for x in labels],capsize=3)
axes[0].set_title('Relative vorticity evolution across quarter-length intervals')
axes[0].set_ylabel('||omega(t+0.24)-omega(t)|| / ||omega(t)||')
axes[1].bar(labels,[stats[x][2] for x in labels])
axes[1].set_title('Temporal field correlation')
axes[1].set_ylabel('Mean spatial Pearson correlation')
for ax in axes:
    ax.tick_params(axis='x',rotation=25)
    ax.grid(axis='y',alpha=.25)
plt.show()
print('INTERPRETATION: a later start does not automatically imply greater change.')
print('Before generating, choose START_TIME in cell 1 and set RUN_GENERATION=True.')


In [ ]:
#@title 3. Reuse verified ETDRK4 GPU solver at finer time step

def seed_vorticity(index,n=NX):
    """Same IC family/seed convention as the uploaded NumPy CPU generator."""
    rng=np.random.default_rng(BASE_SEED+9973*int(index))
    eta=rng.standard_normal((n,n))
    freq=2*np.pi*np.fft.fftfreq(n,d=DOMAIN/n)
    kx,ky=np.meshgrid(freq,freq,indexing='ij')
    K=np.sqrt(kx*kx+ky*ky)
    filt=np.exp(-(K/IC_FILTER_K0)**4)
    filt[0,0]=0.0
    dealias=(np.abs(kx)<=n//3)*(np.abs(ky)<=n//3)
    w=np.fft.ifft2(np.fft.fft2(eta)*filt*dealias).real
    w-=w.mean()
    st=w.std()
    if st>0:w=(IC_AMPLITUDE/st)*w
    return w.astype(np.float32)


def phi_series(z,j,terms=32):
    """phi_j(z) = sum(z**k/(k+j)!, k>=0); stable at z=0.

    z real float64, j=1,2,3. This avoids catastrophic cancellation.
    For this benchmark max |dt*nu*k²| <=~0.33 even at 256².
    """
    term=torch.ones_like(z)/math.factorial(j)
    result=term.clone()
    for k in range(1,terms+1):
        term=term*z/(k+j)
        result=result+term
    return result

class KolmogorovETDRK4:
    def __init__(self,n=128,dt=0.005,dtype=torch.float32,device=DEVICE):
        assert n%2==0 and n>=16 and n%4==0
        self.n=n; self.dt=dt; self.dtype=dtype; self.device=device
        self.cd=torch.complex64 if dtype==torch.float32 else torch.complex128
        k=2*np.pi*torch.fft.fftfreq(n,d=float(DOMAIN)/n,device=device,dtype=torch.float64)
        kx,ky=torch.meshgrid(k,k,indexing='ij')
        k2=kx*kx+ky*ky
        cut=n//3
        self.cut=cut
        mask=(torch.abs(kx)<=cut)&(torch.abs(ky)<=cut)
        self.mask=mask.to(self.cd)
        self.kx=kx.to(dtype);self.ky=ky.to(dtype)
        self.safe_k2=k2.clamp_min(1.0).to(dtype)
        self.safe_k2[0,0]=1.
        self.kmag=torch.sqrt(k2).to(dtype)
        y=(torch.arange(n,device=device,dtype=dtype)*float(DOMAIN)/n)[None,:]
        source=-FORCING_MODE*torch.cos(FORCING_MODE*y).expand(n,-1)
        self.source_hat=(torch.fft.fft2(source)*self.mask).to(self.cd)
        self.source_hat[0,0]=0
        z=-NU*k2*dt
        self.E=torch.exp(z).to(dtype)
        self.E2=torch.exp(z/2).to(dtype)
        p1,p2,p3=(phi_series(z,i) for i in (1,2,3))
        self.Q=(.5*dt*phi_series(z/2,1)).to(dtype)
        self.F1=(dt*(p1-3*p2+4*p3)).to(dtype)
        self.F2=(dt*(p2-2*p3)).to(dtype)
        self.F3=(dt*(4*p3-p2)).to(dtype)
        # At zero frequency RK4 coefficients must each equal dt/6.
        assert abs(float(self.F1[0,0])-dt/6)<1e-7
        assert abs(float(self.F2[0,0])-dt/6)<1e-7
        assert abs(float(self.F3[0,0])-dt/6)<1e-7

    def to_spectral(self,w):
        z=torch.fft.fft2(w.to(device=self.device,dtype=self.dtype)) * self.mask
        z[...,0,0]=0
        return z

    def physical(self,z):
        return torch.fft.ifft2(z).real

    def velocities(self,z):
        psi=z/self.safe_k2
        u=torch.fft.ifft2((1j*self.ky)*psi).real
        v=torch.fft.ifft2((-1j*self.kx)*psi).real
        return u,v

    def nonlinearity(self,z):
        z=z*self.mask
        u,v=self.velocities(z)
        dx=torch.fft.ifft2((1j*self.kx)*z).real
        dy=torch.fft.ifft2((1j*self.ky)*z).real
        nonlinear=(-torch.fft.fft2(u*dx+v*dy)+self.source_hat)*self.mask
        nonlinear[...,0,0]=0
        return nonlinear

    def step(self,z):
        Nv=self.nonlinearity(z)
        a=self.E2*z+self.Q*Nv
        Na=self.nonlinearity(a)
        b=self.E2*z+self.Q*Na
        Nb=self.nonlinearity(b)
        c=self.E2*a+self.Q*(2*Nb-Nv)
        Nc=self.nonlinearity(c)
        result=(self.E*z+self.F1*Nv+2*self.F2*(Na+Nb)+self.F3*Nc)*self.mask
        result[...,0,0]=0
        return result

    def cfl(self,z):
        u,v=self.velocities(z)
        factor=self.dt/(DOMAIN/self.n)
        # Local 2D advective CFL: both velocity components at the SAME grid point.
        # max|u| + max|v| taken separately is an upper bound and can
        # significantly overestimate the actual max local advection.
        return float(((u.abs()+v.abs()).amax(dim=(-2,-1))).max().item()*factor)

    def shell_energy_fraction(self,z,lower_fraction=0.8):
        power=z.abs().square().sum(dim=0)
        region=(torch.maximum(self.kx.abs(),self.ky.abs())>=lower_fraction*self.cut)
        return float((power[region].sum()/power.sum().clamp_min(1e-30)).item())

    @torch.no_grad()
    def advance(self,z,steps,monitor=False,check_every=250):
        maxcfl=0.0
        warned=False
        for k in range(1,steps+1):
            z=self.step(z)
            if monitor and (k==1 or k%check_every==0 or k==steps):
                assert bool(torch.isfinite(z).all()), f'Nonfinite spectral state at step {k}'
                cfl=self.cfl(z)
                maxcfl=max(maxcfl,cfl)
                if cfl>CFL_ADVISORY_THRESHOLD and not warned:
                    print(f'[CFL ADVISORY] observed local CFL={cfl:.3f} > '
                          f'{CFL_ADVISORY_THRESHOLD:.2f}; continuing to numerical '
                          'time/space-refinement checks, not assuming stability.',
                          flush=True)
                    warned=True
                if cfl>MAX_RECORDED_CFL:
                    raise RuntimeError(f'Local CFL={cfl:.3f} exceeds HARD numerical '
                                       f'guard {MAX_RECORDED_CFL}. Reduce DT '
                                       'and rerun verification; do not just '
                                       'increase the hard safety threshold.')
        return z,maxcfl


def spectral_resample(w,target_n):
    """FFT-orthogonal interpolation / restriction preserving Fourier amplitudes.

    Input assumed real-valued and bandlimited below old Nyquist. We use
    fft2(..., norm='forward'), centered Fourier-domain pad/crop, and
    ifft2(..., norm='forward') so both grid sizes represent same field.
    """
    old=w.shape[-1]
    assert old==w.shape[-2] and old%2==0 and target_n%2==0
    z=torch.fft.fftshift(torch.fft.fft2(w,norm='forward'),dim=(-2,-1))
    if target_n>old:
        p=(target_n-old)//2
        z=torch.nn.functional.pad(z,(p,p,p,p))
    elif target_n<old:
        p=(old-target_n)//2
        z=z[...,p:p+target_n,p:p+target_n]
    return torch.fft.ifft2(torch.fft.ifftshift(z,dim=(-2,-1)),norm='forward').real


def rel_field(a,b):
    """Combined relative field L2 over all samples, nodes, time."""
    return float((torch.linalg.vector_norm((a-b).reshape(-1).double())/
                 torch.linalg.vector_norm(b.reshape(-1).double()).clamp_min(1e-20)).item())

print('GPU solver ready. No data generated by this cell.')

In [ ]:
#@title 4. Verify integration from stored anchor and half-step replay agreement
# This cell runs only when RUN_GENERATION=True, so diagnostics are free of regeneration.
if RUN_GENERATION:
    engine=KolmogorovETDRK4(NX,DT,torch.float32,DEVICE)
    @torch.no_grad()
    def integrate(ids,return_trajectory=False):
        ids=list(ids)
        arr=torch.stack([U0[i,:,:,ANCHOR_IDX] for i in ids]).to(DEVICE)
        z=engine.to_spectral(arr)
        cfl=engine.cfl(z)
        # At 0.25, start from original t=0.24 anchor, integrate 4 microsteps.
        for step in range(WARM_STEPS):
            z=engine.step(z)
        assert torch.isfinite(z).all()
        start=engine.physical(z)
        frames=[start.detach().cpu().numpy()] if return_trajectory else None
        errors=[]
        # When absolute time lies on old 0.02 snapshot grid, evaluate replay.
        for k in range(N_STEPS+1):
            absolute=START_TIME+k*DT
            j=int(round(absolute/OLD_SNAPSHOT_DT))
            if j<51 and abs(absolute-j*OLD_SNAPSHOT_DT)<1e-9:
                predicted=engine.physical(z).cpu()
                truth=torch.stack([U0[i,:,:,j] for i in ids])
                dist=torch.linalg.vector_norm((predicted-truth).reshape(len(ids),-1).double(),dim=1)
                norm=torch.linalg.vector_norm(truth.reshape(len(ids),-1).double(),dim=1).clamp_min(1e-12)
                for idx,val in zip(ids,(dist/norm).tolist()):
                    errors.append((idx,absolute,float(val)))
            if k==N_STEPS:break
            z=engine.step(z)
            if (k+1)%10==0 or k==N_STEPS-1:
                assert torch.isfinite(z).all(),'Nonfinite spectral state'
                cfl=max(cfl,engine.cfl(z))
                assert cfl<MAX_RECORDED_CFL,(
                    f'Hard local CFL {cfl:.3f} exceeded; do not relax accuracy gates')
            if return_trajectory:
                frames.append(engine.physical(z).cpu().numpy())
        return (np.stack(frames,axis=-1) if return_trajectory else None),errors,cfl
    test_ids=[0,57,199,200,229,230,299]
    _,replay,cfl=integrate(test_ids)
    assert replay,'No overlapping parent snapshots to compare'
    maximum=max(e[2] for e in replay)
    print(f'[REPLAY] {len(test_ids)} trajectory IDs; {len(replay)} overlaps; '
          f'max relative L2={maximum:.4%}; local CFL max={cfl:.3f}')
    assert maximum<REL_REPLAY_TOL,(
        f'Half-step solver disagrees with verified source ({maximum:.4%}). '
        'Do not generate dataset before investigating the mismatch.')
    print('[PILOT PASS] Short-horizon restart agrees with original verified data.')
else:
    print('Generation disabled. Review quarter-interval diagnostics first.')


In [ ]:
#@title 5. Generate/reuse hashed GPU batches and write to Drive
if RUN_GENERATION:
    assert replay and max(e[2] for e in replay)<REL_REPLAY_TOL
    specfile=ROOT/'dataset_specification.json'
    if specfile.is_file():
        old=json.loads(specfile.read_text())
        assert old['spec_hash']==SPEC_HASH and old['spec']==SPEC, (
            'Output folder contains a different experiment; choose a new dataset folder')
    else:
        specfile.write_text(json.dumps({'spec_hash':SPEC_HASH,'spec':SPEC},indent=2))
    def checksum_file(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as f:
            while True:
                b=f.read(8*1024*1024)
                if not b:break
                h.update(b)
        return h.hexdigest()
    def chunk_files(begin,end):
        stem=f'{SPEC_HASH}_indices{begin:03d}-{end-1:03d}'
        return CHUNKS/(stem+'.npy'),CHUNKS/(stem+'.json')
    for begin in range(0,300,BATCH_SIZE):
        end=min(begin+BATCH_SIZE,300)
        dest,metadata=chunk_files(begin,end)
        if dest.exists() and metadata.exists():
            info=json.loads(metadata.read_text())
            assert info['hash']==SPEC_HASH and info['begin']==begin and info['end']==end
            assert checksum_file(dest)==info['sha256']
            print('[RESUME]',begin,'to',end,flush=True)
            continue
        assert not dest.exists() and not metadata.exists(),(
            'Incomplete chunk; inspect before proceeding: '+str(dest))
        tic=time.perf_counter()
        data,overlap_errors,peak_cfl=integrate(range(begin,end),True)
        assert data.shape==(end-begin,NX,NX,N_FRAMES)
        assert np.isfinite(data).all()
        assert overlap_errors,'No parent overlapping times'
        worst=max(x[2] for x in overlap_errors)
        assert worst<REL_REPLAY_TOL,f'Parent replay failed in chunk {begin}:{end}: {worst}'
        mean_error=np.max(np.abs(data.mean(axis=(1,2))))
        assert mean_error<.002, 'Mean vorticity drift too large'
        local=LOCAL/dest.name
        np.save(local,data,allow_pickle=False)
        checksum=checksum_file(local)
        shutil.copy2(local,dest)
        assert checksum_file(dest)==checksum
        metadata.write_text(json.dumps({
            'hash':SPEC_HASH,'begin':begin,'end':end,'sha256':checksum,
            'shape':list(data.shape),'max_parent_replay_relative_l2':worst,
            'max_local_cfl':peak_cfl,'max_abs_spatial_mean':float(mean_error),
            'elapsed_seconds':time.perf_counter()-tic,
        },indent=2))
        local.unlink()
        del data
        gc.collect()
        print(f'[SAVED] IDs {begin}–{end-1}; replay {worst:.3%}; '
              f'CFL {peak_cfl:.3f}; {time.perf_counter()-tic:.1f}s',flush=True)
    print('All 300 new short trajectories saved to Google Drive (chunk format).')
else:
    print('Generation disabled; no files created. Set RUN_GENERATION=True in cell 1.')


In [ ]:
#@title 6. Verify complete dataset, export .pt, preserve original data
if RUN_GENERATION:
    output_npy=LOCAL/'kolmogorov_short.npy'
    output=np.lib.format.open_memmap(output_npy,mode='w+',dtype=np.float32,shape=OUTPUT_SHAPE)
    chunk_meta=[]
    for begin in range(0,300,BATCH_SIZE):
        end=min(begin+BATCH_SIZE,300)
        file,meta=chunk_files(begin,end)
        assert file.is_file() and meta.is_file()
        info=json.loads(meta.read_text())
        assert checksum_file(file)==info['sha256']
        block=np.load(file,mmap_mode='r',allow_pickle=False)
        assert tuple(block.shape)==(end-begin,NX,NX,N_FRAMES)
        assert np.isfinite(block).all()
        output[begin:end]=block
        chunk_meta.append(info)
    output.flush()
    indices={
        'train_indices':torch.arange(0,200,dtype=torch.long),
        'val_indices':torch.arange(200,230,dtype=torch.long),
        'test_indices':torch.arange(230,300,dtype=torch.long),
    }
    # Verify again: every chunk was checked against the source at common times.
    assert all(x['max_parent_replay_relative_l2']<REL_REPLAY_TOL for x in chunk_meta)
    assert len(chunk_meta)==math.ceil(300/BATCH_SIZE)
    assert np.isfinite(output).all()
    assert np.max(np.abs(output.mean(axis=(1,2))))<.002
    truth=torch.from_numpy(output)
    dataset={
        'U':truth,'omega':truth,
        'x':torch.from_numpy(X.astype(np.float32)),
        'y':torch.from_numpy(Y.astype(np.float32)),
        'times':torch.from_numpy(TIMES.astype(np.float32)),
        'absolute_times':torch.from_numpy(ABSOLUTE_TIMES.astype(np.float32)),
        **indices,
        'Re':RE,'nu':NU,'forcing_mode':FORCING_MODE,
        'dt':DT,'snapshot_dt':DT,'T_spinup':T_SPINUP,
        'T_forecast':HORIZON,'forecast_start':START_TIME,
        'axis_order':'sample, x, y, time',
        'parent_dataset_sha256':EXPECTED_PARENT_SHA,'spec_hash':SPEC_HASH,
    }
    local_pt=LOCAL/'kolmogorov_dataset.pt'
    torch.save(dataset,local_pt)
    del dataset,truth,output
    gc.collect()
    try:
        check=torch.load(local_pt,map_location='cpu',weights_only=True,mmap=True)
    except TypeError:
        check=torch.load(local_pt,map_location='cpu',weights_only=True)
    assert tuple(check['U'].shape)==OUTPUT_SHAPE
    assert check['U'].data_ptr()==check['omega'].data_ptr()
    assert abs(float(check['times'][-1])-.25)<1e-6
    np.testing.assert_array_equal(check['train_indices'].numpy(),np.arange(0,200))
    np.testing.assert_array_equal(check['val_indices'].numpy(),np.arange(200,230))
    np.testing.assert_array_equal(check['test_indices'].numpy(),np.arange(230,300))
    assert bool(torch.isfinite(check['U']).all())
    del check
    digest=checksum_file(local_pt)
    target=ROOT/'kolmogorov_dataset.pt'
    if target.is_file():
        assert checksum_file(target)==digest,(
            'Existing Drive .pt differs. Do not silently overwrite.')
        print('[RESUME] Verified matching final dataset')
    else:
        partial=ROOT/'kolmogorov_dataset.pt.partial'
        if partial.exists():partial.unlink()
        shutil.copy2(local_pt,partial)
        assert checksum_file(partial)==digest
        partial.replace(target)
        assert checksum_file(target)==digest
    report={
        'status':'PASS','spec':SPEC,'spec_hash':SPEC_HASH,
        'parent_sha256':EXPECTED_PARENT_SHA,
        'sha256':digest,'shape':list(OUTPUT_SHAPE),
        'max_parent_replay_relative_l2':max(x['max_parent_replay_relative_l2'] for x in chunk_meta),
        'max_local_cfl':max(x['max_local_cfl'] for x in chunk_meta),
        'chunks':chunk_meta,
        'training_count':200,'validation_count':30,'test_count':70,
    }
    (ROOT/'metadata.json').write_text(json.dumps(report,indent=2))
    print('\nDATASET VERIFIED:',target)
    print('Shape:',OUTPUT_SHAPE,'| size:',round(target.stat().st_size/2**30,3),'GiB')
    print('Duration:',HORIZON,'| frames:',N_FRAMES,'| saved dt:',DT)
    print('SHA256:',digest,'| parent source:',EXPECTED_PARENT_SHA)
    del local_pt
else:
    print('No final dataset produced: RUN_GENERATION=False.')


In [ ]:
#@title 7. Visualize initial and final short-horizon vorticity
if RUN_GENERATION and (LOCAL/'kolmogorov_short.npy').exists():
    arr=np.load(LOCAL/'kolmogorov_short.npy',mmap_mode='r')
    fig,axs=plt.subplots(2,4,figsize=(14,7),constrained_layout=True)
    for row,idx in enumerate([0,230]):
        for col,j in enumerate([0,25,50,100]):
            field=np.asarray(arr[idx,:,:,j])
            vlim=max(abs(float(arr[idx].min())),abs(float(arr[idx].max())))
            axs[row,col].imshow(field.T,origin='lower',extent=[0,2*np.pi,0,2*np.pi],
                                cmap='RdBu_r',vmin=-vlim,vmax=vlim)
            axs[row,col].set_title(f'ID {idx} at t={ABSOLUTE_TIMES[j]:.3f}')
    path=ROOT/'preview_short_horizon.png'
    fig.savefig(path,dpi=150)
    plt.show()
    print('Preview:',path)
else:
    print('Generate the dataset first to see spatial snapshots.')


In [ ]:
#@title 8. QUALITY AUDIT: 101-frame dataset (no new simulation or GPU training)
from pathlib import Path
import numpy as np, torch, json, hashlib
import matplotlib.pyplot as plt

# Locate the actually saved dataset. START_TIME in the setup cell may have
# been reset to 0.25 after generating with a different start time.
# Also distinguish COMPLETE generation chunks from a finalized .pt file.
QUALITY_DATASET_FOLDER='' #@param {type:'string'}
quality_root=Path(QUALITY_DATASET_FOLDER) if QUALITY_DATASET_FOLDER.strip() else ROOT
quality_path=quality_root/'kolmogorov_dataset.pt'
if not quality_path.is_file():
    base=Path('/content/drive/MyDrive')
    folders=sorted(base.glob('Kolmogorov_NS_Re500_start*_horizon*_dt*_v1'))
    available=[]
    print('[DATASET SEARCH] Expected file is missing:',quality_path)
    for folder in folders:
        final_file=folder/'kolmogorov_dataset.pt'
        meta_file=folder/'metadata.json'
        config_file=folder/'dataset_specification.json'
        chunks_dir=folder/'chunks'
        chunk_files=sorted(chunks_dir.glob('*.npy')) if chunks_dir.is_dir() else []
        status='FINALIZED' if final_file.is_file() and meta_file.is_file() else 'CHUNKS ONLY' if chunk_files else 'INCOMPLETE'
        print(' ',folder.name,'->',status,
              '| batch files:',len(chunk_files),
              '| dataset_specification:',config_file.is_file())
        if config_file.is_file():
            info=json.loads(config_file.read_text())
            original_spec=info.get('spec',{})
            print('    start_time=',original_spec.get('start_time'),
                  ' batch_size=',original_spec.get('batch_size'),
                  ' spec_hash=',info.get('spec_hash'))
        if final_file.is_file() and meta_file.is_file():
            try:
                candidate_meta=json.loads(meta_file.read_text())
                if (candidate_meta.get('status')=='PASS'
                    and candidate_meta.get('parent_sha256')==EXPECTED_PARENT_SHA
                    and candidate_meta.get('shape')==[300,128,128,101]):
                    available.append(folder)
            except (ValueError, OSError):
                pass
    if not QUALITY_DATASET_FOLDER.strip() and len(available)==1:
        quality_root=available[0]
        quality_path=quality_root/'kolmogorov_dataset.pt'
        print('[FOUND] Using the single verified 101-frame dataset:',quality_path)
    elif len(available)>1:
        raise FileNotFoundError(
            'More than one verified 101-frame dataset exists. Set '
            'QUALITY_DATASET_FOLDER above to the exact Drive folder and rerun.')
    else:
        raise FileNotFoundError(
            'No finalized, verified 101-frame .pt dataset was found. If CHUNKS ONLY '
            'appears above, generation batches survived but final assembly '
            '(cell 6) has not been completed. Restore the START_TIME and BATCH_SIZE '
            'shown in dataset_specification.json, set RUN_GENERATION=True in '
            'cell 1, then run cells 1, 3, 4, 5 and 6. Existing checked chunks '
            'are reused; the 20-unit spin-up is NOT rerun. Do not repeat '
            'GPU trajectory generation or delete any chunks.')

quality_meta_path=quality_root/'metadata.json'
assert quality_meta_path.is_file(), 'Final dataset found without metadata: '+str(quality_meta_path)
quality_meta=json.loads(quality_meta_path.read_text())
assert quality_meta.get('status')=='PASS', 'Short-horizon verification did not PASS'
assert quality_meta['parent_sha256']==EXPECTED_PARENT_SHA
assert quality_meta['shape']==[300,128,128,101]
print('[QUALITY SOURCE] Using:',quality_root)

try:
    quality_data=torch.load(quality_path, map_location='cpu', weights_only=True, mmap=True)
except TypeError:
    quality_data=torch.load(quality_path, map_location='cpu', weights_only=True)
Q=quality_data['U']
assert Q.shape==(300,128,128,101) and Q.dtype==torch.float32
qt=np.asarray(quality_data['times'],dtype=np.float64)
assert len(qt)==101 and np.allclose(np.diff(qt),0.0025,atol=3e-8)
assert abs(qt[-1]-.25)<1e-7
for key,a,b in [('train_indices',0,200),('val_indices',200,230),('test_indices',230,300)]:
    np.testing.assert_array_equal(np.asarray(quality_data[key]), np.arange(a,b))
assert quality_data['parent_dataset_sha256']==EXPECTED_PARENT_SHA
assert quality_meta['spec_hash']==quality_data['spec_hash'], 'Metadata and dataset config mismatch'
assert abs(float(quality_data['forecast_start']) -
           float(quality_meta['spec']['start_time']))<1e-8, 'Start time mismatch'
print('='*68)
print('NAVIER-STOKES DATASET QUALITY — 101 SNAPSHOTS')
print('='*68)
print('File:',quality_path)
print('Shape:',tuple(Q.shape),'relative times:',qt[0],qt[-1])
print('Absolute times:',float(quality_data['forecast_start']), 'to',
      float(quality_data['forecast_start'])+float(qt[-1]))
print('Train/val/test: 200/30/70; initial-state provenance:',EXPECTED_PARENT_SHA)
print('Generation max source-replay error:',
      max(float(c['max_parent_replay_relative_l2']) for c in quality_meta['chunks']))
print('Generation max local CFL:',quality_meta['max_local_cfl'])
print('Saved SHA256:',quality_meta['sha256'])

# Integrity of ALL 300 samples; not used to choose model hyperparameters.
hashes=set(); max_mean=0.; min_std=float('inf')
for i in range(300):
    w=np.asarray(Q[i],dtype=np.float32)
    assert np.isfinite(w).all(), f'NaN or Inf in sample {i}'
    hash0=hashlib.sha256(np.ascontiguousarray(w[...,0]).tobytes()).hexdigest()
    assert hash0 not in hashes, f'Duplicate initial state in sample {i}'
    hashes.add(hash0)
    max_mean=max(max_mean,float(np.abs(np.mean(w,axis=(0,1),dtype=np.float64)).max()))
    min_std=min(min_std,float(np.std(w,axis=(0,1),dtype=np.float64).min()))
print('\n1. INTEGRITY — ALL 300 TRAJECTORIES')
print('Finite: PASS | Duplicate initial fields: 0')
print('Maximum absolute spatial mean:',f'{max_mean:.3e}')
print('Minimum spatial standard deviation:',f'{min_std:.3e}')
assert max_mean<.002 and min_std>1e-6

# Dynamics/model-selection diagnostics only on training+validation trajectories.
ids=np.unique(np.r_[np.linspace(0,199,70,dtype=int),
                    np.linspace(200,229,20,dtype=int)])
changes=[];steps=[];starts=[];ends=[];freq=[];rms=[]
k=np.fft.fftfreq(128)*128
wave=np.maximum(np.abs(k[:,None]),np.abs(k[None,:]))
cutoffs=[16,24,32,40,43]
for i in ids:
    w=np.asarray(Q[int(i)],dtype=np.float32)
    a=w[...,0].copy();b=w[...,-1].copy()
    starts.append(a.ravel());ends.append(b.ravel())
    base=np.maximum(np.sum(a.astype(np.float64)**2),1e-20)
    changes.append(np.sqrt(np.sum((w-a[...,None]).astype(np.float64)**2,axis=(0,1))/base))
    dw=w[...,1:]-w[...,:-1]
    local_norm=np.sum(w[...,:-1].astype(np.float64)**2,axis=(0,1))
    steps.append(np.sqrt(np.sum(dw.astype(np.float64)**2,axis=(0,1))/
                         np.maximum(local_norm,1e-20)))
    fractions=[]
    for t in [0,50,100]:
        power=np.abs(np.fft.fft2(w[...,t]))**2
        total=float(power.sum())+1e-30
        fractions.append([float(power[wave>=z].sum()/total) for z in cutoffs])
    freq.append(fractions)
    rms.append([float(np.sqrt(np.mean(w[...,j].astype(np.float64)**2)))
                for j in [0,50,100]])
changes=np.asarray(changes);steps=np.asarray(steps)
freq=np.asarray(freq);rms=np.asarray(rms)
def corrs(matrix):
    X=np.stack(matrix).astype(np.float64)
    X-=X.mean(axis=1,keepdims=True)
    X/=np.linalg.norm(X,axis=1,keepdims=True).clip(1e-20)
    C=X@X.T
    upper=C[np.triu_indices(len(C),k=1)]
    p=np.unravel_index(np.argmax(np.abs(C-np.eye(len(C)))),C.shape)
    return C,{'signed_mean':float(upper.mean()),
              'mean_abs':float(np.abs(upper).mean()),
              'p95_abs':float(np.quantile(np.abs(upper),.95)),
              'max_abs':float(np.abs(upper).max())}
C0,c0=corrs(starts);CT,ct=corrs(ends)
def stat(v):
    return {'mean':float(np.mean(v)), 'median':float(np.median(v)),
            'min':float(np.min(v)), 'max':float(np.max(v))}
print('\n2. TRAJECTORY DIVERSITY — TRAIN+VALIDATION')
print('Initial field correlation:',c0)
print('Final field correlation:  ',ct)
print('\n3. TEMPORAL DYNAMICS — TRAIN+VALIDATION')
print('Relative change by T=0.125:',stat(changes[:,50]))
print('Relative change by T=0.250:',stat(changes[:,-1]))
print('Adjacent-snapshot relative change:',stat(steps))
print('RMS vorticity at t=0, .125, .25:',rms.mean(axis=0))
print('\n4. FOURIER CONTENT — TRAIN+VALIDATION')
for j,label in [(0,'start'),(1,'midpoint'),(2,'final')]:
    print(label, {cut:round(float(freq[:,j,q].mean()*100),5)
                  for q,cut in enumerate(cutoffs)}, '(percent energy beyond cutoff)')

# A fresh comparison to the verified parent at matching physical times.
# Do not use the test set for selecting the forecast interval or AWFNO settings.
errors=[]
for i in [0,25,57,100,199,200,229]:
    for j in range(101):
        absolute=float(quality_data['forecast_start'])+float(qt[j])
        parent_index=round(absolute/.02)
        if parent_index>50 or abs(absolute-parent_index*.02)>1e-7:continue
        w=np.asarray(Q[i,...,j],dtype=np.float64)
        parent=np.asarray(U0[i,...,parent_index],dtype=np.float64)
        errors.append(float(np.linalg.norm(w-parent)/
                            max(np.linalg.norm(parent),1e-20)))
assert errors
print('\n5. INDEPENDENT REPLAY AGAINST PARENT DATA (TRAIN+VAL)')
print('Relative L2:',stat(errors))
assert max(errors)<.005, 'Parent replay exceeded 0.5% threshold'

quality_summary={
    'file':str(quality_path),'shape':[300,128,128,101],
    'forecast_start':float(quality_data['forecast_start']),
    'mean_vorticity_abs_max':max_mean,'minimum_spatial_std':min_std,
    'initial_correlation':c0,'final_correlation':ct,
    'change_by_0.125':stat(changes[:,50]), 'change_by_0.25':stat(changes[:,-1]),
    'adjacent_snapshot_relative_change':stat(steps),
    'spectral_energy_fraction':{label:{str(cut):float(freq[:,j,q].mean())
        for q,cut in enumerate(cutoffs)}
        for j,label in [(0,'start'),(1,'midpoint'),(2,'final')]},
    'parent_replay_relative_l2':stat(errors),
    'generator_sha256':quality_meta['sha256'],
    'diagnostics_on':'training+validation only; test used for integrity checks only',
}
(quality_root/'quality_audit.json').write_text(json.dumps(quality_summary,indent=2))
fig,axs=plt.subplots(2,3,figsize=(14,8),constrained_layout=True)
for curve in changes[::5]:
    axs[0,0].plot(qt,curve,alpha=.18)
axs[0,0].plot(qt,changes.mean(axis=0),color='black',lw=2)
axs[0,0].set_title('Relative change from initial field')
axs[0,1].plot(qt[1:],np.median(steps,axis=0))
axs[0,1].set_title('Adjacent-state relative change')
for t,label in enumerate(['initial','middle','final']):
    axs[0,2].plot(cutoffs,np.maximum(freq[:,t,:].mean(axis=0),1e-15),'-o',label=label)
axs[0,2].set_yscale('log')
axs[0,2].legend()
axs[0,2].set_title('Fraction of energy above Fourier cutoff')
axs[1,0].imshow(C0,vmin=-1,vmax=1,cmap='coolwarm')
axs[1,0].set_title('Initial correlations')
axs[1,1].imshow(CT,vmin=-1,vmax=1,cmap='coolwarm')
axs[1,1].set_title('Final correlations')
axs[1,2].plot([0,.125,.25],rms.mean(axis=0),'-o')
axs[1,2].set_title('RMS vorticity')
for ax in axs.flat:ax.grid(alpha=.15)
plt.savefig(quality_root/'quality_audit.png',dpi=150)
plt.show()

fig,axs=plt.subplots(2,5,figsize=(15,6),constrained_layout=True)
for row,i in enumerate([0,200]):
    scale=max(float(np.abs(np.asarray(Q[i,...,t])).max()) for t in [0,25,50,75,100])
    for col,t in enumerate([0,25,50,75,100]):
        axs[row,col].imshow(np.asarray(Q[i,...,t]).T,cmap='RdBu_r',origin='lower',
                            vmin=-scale,vmax=scale)
        axs[row,col].set_title(f'ID {i}, t={float(quality_data["forecast_start"])+t*.0025:.3f}')
        axs[row,col].set_xticks([]);axs[row,col].set_yticks([])
plt.savefig(quality_root/'quality_snapshots.png',dpi=150)
plt.show()
print('\nAUDIT COMPLETE. Results saved to:',quality_root/'quality_audit.json')
print('Figures:',quality_root/'quality_audit.png',quality_root/'quality_snapshots.png')
